# References

https://ogunlao.github.io/blog/2021/01/26/how-to-create-speech-dataset.html

https://mfa-models.readthedocs.io/en/latest/index.html

https://chenzixu.rbind.io/resources/1forcedalignment/fa6/

# Audio Alignment & Segmentation with Montreal Forced Aligner

This notebook demonstrates how to use Montreal Forced Aligner (MFA) to:
1. Align long audio files with their transcriptions
2. Generate accurate word-level timestamps
3. Segment audio into shorter clips with precise timing

**Prerequisites:**
- Audio files (.mp3) in one folder with matching transcription names
- Transcription files (.doc) in another folder with matching audio names
- Same filename pairs (e.g., `interview_001.mp3` and `interview_001.doc`)

## Step 1: Install and Setup Montreal Forced Aligner

First, we'll install MFA and required dependencies. This is commented out because all the libraries were already available in the miniconda environment.

In [ ]:
# import subprocess
# import sys
# import os

# # Install Montreal Forced Aligner via conda
# # If you prefer pip, comment out the conda line and use the pip alternative below

# print("Installing Montreal Forced Aligner and dependencies...")

# # Conda installation (recommended - more stable)
# subprocess.check_call([sys.executable, "-m", "pip", "install", "montreal-forced-aligner"])

# # Additional dependencies needed
# subprocess.check_call([sys.executable, "-m", "pip", "install", 
#                        "librosa",      # Audio processing
#                        "pydub",        # Audio manipulation
#                        "python-docx"   # Reading .doc files
#                       ])

# print("✓ Installation complete!")

## Step 2: Import Libraries and Configure Paths

Set up your audio and transcription folders here.

In [ ]:
import os
import json
import librosa
import soundfile as sf
from pathlib import Path
from collections import defaultdict
import pandas as pd
import re
from num2words import num2words
from docx import Document
import shutil
import numpy as np
import noisereduce as nr
from pydub import AudioSegment, effects
import subprocess
import time
import textgrid
import difflib
from typing import Tuple


# ==================== CONFIGURATION ====================
# TODO: Update these paths to match your folder locations

AUDIO_FOLDER = "/Users/rodrigocarrillo/Documents/Natural Language Processing Projects/Videos Entrevista Medical Spanish/00.Audios/"      # Folder with .mp3 files (your audios should be here)
TRANSCRIPTION_FOLDER = "/Users/rodrigocarrillo/Documents/Natural Language Processing Projects/Videos Entrevista Medical Spanish/01.Transcripciones Humanas/Post-procesado RMCL/"  # Folder with .doc files (your transcriptions should be here)
OUTPUT_FOLDER = "/Users/rodrigocarrillo/Downloads/Output"                              # Where to save aligned data
SEGMENTS_FOLDER = "/Users/rodrigocarrillo/Downloads/Output/Segmented_audio"            # Where to save audio clips


# ==================== SETUP ====================
# Create output folders if they don't exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(SEGMENTS_FOLDER, exist_ok=True)

print(f"Audio folder: {AUDIO_FOLDER}")
print(f"Transcription folder: {TRANSCRIPTION_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")
print(f"Segments folder: {SEGMENTS_FOLDER}")
print("\n✓ Paths configured successfully!")

## Step 3: Load and Process Transcriptions from .doc Files

Extract text from your Word documents and clean them for alignment.

In [ ]:
def clean_transcription(text):
    """
    Cleans the text for MFA, removes punctuation marks,
    normalizes the text, and converts numbers to words.
    """
    if not text:
        return ""

    # 1. Convert to lowercase.
    text = text.lower()

    # 3. Convert numbers to words (e.g., "21" -> "twenti-one")
    def convertir_numeros(match):
        return num2words(int(match.group()), lang='es')
    
    text = re.sub(r'\d+', convertir_numeros, text)

    # 4. Remove punctuation and special characters
    # Keep letters (including accents and ñ) and spaces.
    # This cleans "?" and "," that caused issues.
    text = re.sub(r'[^\w\sáéíóúñ]', ' ', text)

    # 5. Final cleanup of multiple spaces
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

In [ ]:
def extract_text_from_docx(doc_path):
    """
    Extract all text from a .docx file.
    
    Args:
        doc_path: Path to the .doc or .docx file
        
    Returns:
        String containing all extracted text
    """
    try:
        doc = Document(doc_path)
        text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
        return text
    except Exception as e:
        print(f"Error reading {doc_path}: {e}")
        return ""

# Load all transcriptions and match with audio files
transcription_data = {}

for doc_file in os.listdir(TRANSCRIPTION_FOLDER):
    if doc_file.endswith(('.doc', '.docx')):
        # Get filename without extension
        base_name = os.path.splitext(doc_file)[0]
        
        doc_path = os.path.join(TRANSCRIPTION_FOLDER, doc_file)
        raw_text = extract_text_from_docx(doc_path)
        cleaned_text = clean_transcription(raw_text)
        
        transcription_data[base_name] = cleaned_text
        print(f"✓ Loaded: {base_name}")

print(f"\nTotal transcriptions loaded: {len(transcription_data)}")

# Display sample
if transcription_data:
    sample_key = list(transcription_data.keys())[0]
    print(f"\nSample transcription preview ({sample_key}):")
    print(transcription_data[sample_key][:200] + "...")

## Step 4: Prepare Files for Montreal Forced Aligner

MFA requires a specific directory structure. We'll create it with our audio and text files. We will also clean the audio to improve quality.

In [ ]:
def procesar_audio_calidad(y, sr):
    """
    Applies professional noise reduction and normalization.
    """
    # 1. Noise reduction.
    # We use a non-destructive reduction (prop_decrease=0.7) to avoid robotizing the voice
    y_denoised = nr.reduce_noise(y=y, sr=sr, prop_decrease=0.7)
    
    # 2. Convert to pydub for normalization (requires int16)
    y_int = (y_denoised * 32767).astype(np.int16)
    audio_segment = AudioSegment(
        y_int.tobytes(), 
        frame_rate=sr,
        sample_width=2, 
        channels=1
    )
    
    # 3. Volume normalization (Peak Normalization to -1.0 dB)
    normalized_audio = effects.normalize(audio_segment)
    
    # Convert back to float32 to save with soundfile
    y_final = np.array(normalized_audio.get_array_of_samples()).astype(np.float32) / 32768.0
    return y_final

MFA_CORPUS_DIR = os.path.join(OUTPUT_FOLDER, "mfa_corpus")
if os.path.exists(MFA_CORPUS_DIR):
    shutil.rmtree(MFA_CORPUS_DIR)
os.makedirs(MFA_CORPUS_DIR)

matched_files = 0

for audio_file in os.listdir(AUDIO_FOLDER):
    if audio_file.endswith(('.mp3', '.wav', '.flac', '.m4a')):
        base_name = os.path.splitext(audio_file)[0]
        
        if base_name in transcription_data:
            speaker_dir = os.path.join(MFA_CORPUS_DIR, base_name)
            os.makedirs(speaker_dir, exist_ok=True)
            
            src_audio = os.path.join(AUDIO_FOLDER, audio_file)
            dst_audio = os.path.join(speaker_dir, f"{base_name}.wav")
            
            try:
                # A. Load (Force 16k and Mono)
                y, sr = librosa.load(src_audio, sr=16000, mono=True)
                
                # B. IMPROVE (Noise + Volume)
                y_clean = procesar_audio_calidad(y, sr)
                
                # C. SAVE AUDIO (PCM_16)
                sf.write(dst_audio, y_clean, sr, subtype='PCM_16')
                
                # D. SAVE TEXT (Cleaned for MFA: no punctuation, numbers in words)
                txt_file = os.path.join(speaker_dir, f"{base_name}.txt")
                cleaned_text = clean_transcription(transcription_data[base_name])
                
                with open(txt_file, 'w', encoding='utf-8') as f:
                    f.write(cleaned_text)
                
                print(f"✓ Procesado con éxito: {base_name} (Audio Limpio + Texto MFA)")
                matched_files += 1
                
            except Exception as e:
                print(f"❌ Error crítico en {audio_file}: {e}")

print(f"\n--- Preparación Finalizada ---")
print(f"Total: {matched_files} pares listos para MFA Align.")

## Step 5: Download Language Models (Required for First Run)

MFA needs a trained acoustic model and pronunciation dictionary. Download them once here.

In [ ]:
# Language for MFA (English is default, change if needed). Options: "spanish_mfa" "spanish_latin_america_mfa" "mfa_spanish_latin_america_mfa_dictionary_2022"
LANGUAGE_ACUSTIC_MODEL = "spanish_mfa"                 # Only option for spanish.
LANGUAGE_DICTIONARY = "spanish_latin_america_mfa"      # Tried Spain Spanish first, did not work better.
LANGUAGE_G2P = "spanish_latin_america_mfa"             # Tried Spain Spanish first, did not work better.



def ejecutar_mfa(comandos):
    """Ejecuta comandos de MFA de forma segura mediante el sistema."""
    try:
        # Usamos el comando 'mfa' directamente como si fuera la terminal
        proceso = subprocess.run(['mfa'] + comandos, 
                                capture_output=True, 
                                text=True, 
                                check=True)
        print(proceso.stdout)
    except subprocess.CalledProcessError as e:
        # Si el modelo ya existe, MFA suele devolver un aviso
        if "already exists" in e.stderr or "already downloaded" in e.stderr:
            print(f"⚠ El componente ya está disponible localmente.")
        else:
            print(f"❌ Error al ejecutar MFA: {e.stderr}")

print(f"--- Iniciando descarga de modelos para: {LANGUAGE_ACUSTIC_MODEL} & {LANGUAGE_DICTIONARY} & {LANGUAGE_G2P} ---")

# 1. Descargar Modelo Acústico
print(f"\nDescargando modelo acústico ({LANGUAGE_ACUSTIC_MODEL})...")
ejecutar_mfa(['model', 'download', 'acoustic', LANGUAGE_ACUSTIC_MODEL])

# 2. Descargar Diccionario
print(f"\nDescargando diccionario ({LANGUAGE_DICTIONARY})...")
ejecutar_mfa(['model', 'download', 'dictionary', LANGUAGE_DICTIONARY])

# 3. Descargar G2P (Grapheme-to-Phoneme)
print(f"\nDescargando modelo G2P ({LANGUAGE_G2P})...")
ejecutar_mfa(['model', 'download', 'g2p', LANGUAGE_G2P])

print("\n✓ ¡Todos los modelos en español están listos!")

## Step 6: Run Montreal Forced Aligner

This is where the magic happens! MFA aligns your audio with the transcriptions and generates precise timestamps.

⏱️ **Note:** This may take 5-10 minutes depending on audio length. The longer the audio, the longer it takes.

In [ ]:
print("=" * 60)
print("STARTING FORCED ALIGNMENT")
print("=" * 60)
print("\nThis process:")
print("1. Analyzes your audio files")
print("2. Matches them with transcriptions")  
print("3. Generates word-level timestamps")
print("4. Creates JSON files with alignment data\n")

MFA_OUTPUT_DIR = os.path.join(OUTPUT_FOLDER, "mfa_output")

# Prepare the command
# align: command to run alignment
# corpus path: our prepared audio + text files
# language model: acoustic model
# output directory: where to save results

# Parámetros de robustez adicionales
# --beam: Aumenta el ancho de búsqueda (default suele ser 10 o 40)
# --retry_beam: Un segundo intento más profundo si el primero falla
# --overwrite: Asegura que si hubo intentos fallidos previos, los sobrescriba
# --clean: Limpia archivos temporales de sesiones anteriores que pueden causar conflictos
# --use_mp: Usa multiprocesamiento para agilizar la búsqueda profunda

cmd = [
    "mfa",
    "align",
    MFA_CORPUS_DIR,           # Input corpus directory
    LANGUAGE_DICTIONARY,      # Language
    LANGUAGE_ACUSTIC_MODEL,   # Acoustic model
    MFA_OUTPUT_DIR,           # Output directory
    "--beam", "1000",            # Increase the aligner's patience (100 is a good value)
    "--retry_beam", "4000",      # Desperate attempt if the normal beam fails (400 is reasonable)
    "--no_debug",                # Avoid stopping on minor errors
    "--overwrite",               # Overwrite previous results
    "--clean",                   # Clean cache to avoid errors from collapsed sessions
    "--verbose"                  # Will give us more details on why it ignores files                # All these options were implemented because otherwise, the model would only process three audio files. With these options, at least it processed 9 out of 10.
]

print(f"Running command: {' '.join(cmd)}\n")

start_time = time.time()

try:
    # Run the alignment
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    
    elapsed = time.time() - start_time
    
    print("STDOUT:")
    print(result.stdout)
    
    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)
    
    print(f"\n✓ Alignment completed successfully!")
    print(f"Time elapsed: {elapsed/60:.1f} minutes")
    print(f"Results saved to: {MFA_OUTPUT_DIR}")
    
except subprocess.CalledProcessError as e:
    print(f"✗ Error during alignment:")
    print(f"Return code: {e.returncode}")
    print(f"\nSTDOUT:\n{e.stdout}")
    print(f"\nSTDERR:\n{e.stderr}")

## Step 7: Parse Alignment Results

Extract word-level timestamps from the JSON output generated by MFA.

In [ ]:
def extract_word_timestamps(textgrid_path):
    """
    Extract word-level timestamps from MFA output TextGrid file.
    
    Args:
        textgrid_path: Path to the .TextGrid file from MFA
        
    Returns:
        List of dictionaries with word, start_time, and end_time
    """
    try:
        tg = textgrid.TextGrid.fromFile(textgrid_path)
        
        # MFA typically creates a "words" tier
        words_tier = None
        for tier in tg.tiers:
            if 'word' in tier.name.lower():
                words_tier = tier
                break
        
        if words_tier is None:
            print(f"Warning: No 'word' tier found in {textgrid_path}")
            return []
        
        # Extract intervals (words with timestamps)
        word_timestamps = []
        for interval in words_tier:
            if interval.mark.strip():  # Skip empty intervals
                word_timestamps.append({
                    'word': interval.mark,
                    'start': interval.minTime,
                    'end': interval.maxTime,
                    'duration': interval.maxTime - interval.minTime
                })
        
        return word_timestamps
    
    except Exception as e:
        print(f"Error reading {textgrid_path}: {e}")
        return []

# Parse all alignment results
MFA_OUTPUT_DIR = os.path.join(OUTPUT_FOLDER, "mfa_output")
alignment_results = {}

print("Parsing alignment results...\n")

for speaker_dir in os.listdir(MFA_OUTPUT_DIR):
    speaker_path = os.path.join(MFA_OUTPUT_DIR, speaker_dir)
    
    if os.path.isdir(speaker_path):
        # Look for .TextGrid files
        for file in os.listdir(speaker_path):
            if file.endswith('.TextGrid'):
                textgrid_path = os.path.join(speaker_path, file)
                base_name = os.path.splitext(file)[0]
                
                word_times = extract_word_timestamps(textgrid_path)
                
                if word_times:
                    alignment_results[base_name] = word_times
                    print(f"✓ {base_name}: {len(word_times)} words extracted")
                    
                    # Show sample
                    if len(word_times) > 0:
                        sample = word_times[:3]
                        for w in sample:
                            print(f"   [{w['start']:.2f}s - {w['end']:.2f}s] {w['word']}")
                        if len(word_times) > 3:
                            print(f"   ... and {len(word_times) - 3} more words")

print(f"\n✓ Total files aligned: {len(alignment_results)}")

## Step 8: Segment Audio into Shorter Clips

Use the timestamps to extract individual words or phrases as separate audio files.

In [ ]:
def segment_audio_by_words(audio_path, word_timestamps, output_dir, base_name, 
                          include_silence_ms=50):
    """
    Segment audio file into individual word clips.
    
    Args:
        audio_path: Path to the audio file
        word_timestamps: List of word timestamp dicts from extract_word_timestamps()
        output_dir: Directory to save segmented clips
        base_name: Base filename for the output clips
        include_silence_ms: Add silence before and after each word (milliseconds)
    """
    
    # Load audio
    y, sr = librosa.load(audio_path, sr=None)
    
    # Create output directory for this file
    file_segment_dir = os.path.join(output_dir, base_name)
    os.makedirs(file_segment_dir, exist_ok=True)
    
    # Convert silence duration to samples
    silence_samples = int((include_silence_ms / 1000) * sr)
    
    segment_info = []
    
    # Segment each word
    for i, word_data in enumerate(word_timestamps):
        word = word_data['word']
        start_time = word_data['start']
        end_time = word_data['end']
        
        # Add padding (silence)
        start_sample = max(0, int(start_time * sr) - silence_samples)
        end_sample = min(len(y), int(end_time * sr) + silence_samples)
        
        # Extract audio segment
        segment = y[start_sample:end_sample]
        
        # Save segment
        segment_filename = f"{base_name}_{i:04d}_{word}.wav"
        segment_path = os.path.join(file_segment_dir, segment_filename)
        
        sf.write(segment_path, segment, sr)
        
        segment_info.append({
            'segment_num': i,
            'word': word,
            'filename': segment_filename,
            'original_start': start_time,
            'original_end': end_time,
            'segment_start': start_sample / sr,
            'segment_end': end_sample / sr,
            'duration': (end_sample - start_sample) / sr
        })
    
    return segment_info

# Segment audio for all aligned files
print("Segmenting audio files by words...\n")

all_segments_metadata = {}

for base_name, word_times in alignment_results.items():
    # Find the original audio file
    audio_path = None
    
    for audio_file in os.listdir(AUDIO_FOLDER):
        if os.path.splitext(audio_file)[0] == base_name:
            audio_path = os.path.join(AUDIO_FOLDER, audio_file)
            break
    
    if audio_path and os.path.exists(audio_path):
        print(f"Processing: {base_name}")
        
        segment_info = segment_audio_by_words(
            audio_path, 
            word_times, 
            SEGMENTS_FOLDER,
            base_name,
            include_silence_ms=50
        )
        
        all_segments_metadata[base_name] = segment_info
        print(f"  ✓ Created {len(segment_info)} segment files\n")
    else:
        print(f"  ✗ Audio file not found for {base_name}\n")

# Save metadata
metadata_path = os.path.join(SEGMENTS_FOLDER, 'segments_metadata.json')
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(all_segments_metadata, f, indent=2, ensure_ascii=False)

print(f"✓ Segmentation complete!")
print(f"✓ Metadata saved to: {metadata_path}")

## Step 9: Advanced - Segment by Phrases or Time Chunks

Option to group words into phrases or fixed-duration chunks instead of individual words.

In [ ]:
def segment_audio_by_duration(audio_path, word_timestamps, output_dir, base_name,
                              chunk_duration_sec=10):
    """
    Segment audio into fixed-duration chunks, keeping word boundaries intact.
    
    Args:
        audio_path: Path to the audio file
        word_timestamps: List of word timestamp dicts
        output_dir: Directory to save segments
        base_name: Base filename
        chunk_duration_sec: Target duration for each chunk
    """
    
    y, sr = librosa.load(audio_path, sr=None)
    file_segment_dir = os.path.join(output_dir, base_name)
    os.makedirs(file_segment_dir, exist_ok=True)
    
    segment_info = []
    chunk_num = 0
    current_chunk_start = 0
    current_chunk_words = []
    
    for word_data in word_timestamps:
        current_chunk_words.append(word_data)
        chunk_end = word_data['end']
        chunk_duration = chunk_end - current_chunk_start
        
        # If we exceed target duration, save this chunk
        if chunk_duration >= chunk_duration_sec or word_data == word_timestamps[-1]:
            # Extract audio for this chunk
            start_sample = int(current_chunk_start * sr)
            end_sample = int(chunk_end * sr)
            segment = y[start_sample:end_sample]
            
            # Create chunk filename
            chunk_text = " ".join([w['word'] for w in current_chunk_words])
            chunk_filename = f"{base_name}_chunk_{chunk_num:03d}.wav"
            chunk_path = os.path.join(file_segment_dir, chunk_filename)
            
            sf.write(chunk_path, segment, sr)
            
            segment_info.append({
                'chunk_num': chunk_num,
                'filename': chunk_filename,
                'text': chunk_text,
                'start_time': current_chunk_start,
                'end_time': chunk_end,
                'duration': chunk_end - current_chunk_start,
                'word_count': len(current_chunk_words)
            })
            
            # Reset for next chunk
            chunk_num += 1
            current_chunk_start = chunk_end
            current_chunk_words = []
    
    return segment_info

# Example: Segment by 10-second chunks
print("Segmenting audio into 10-second chunks...\n")

chunk_segments_metadata = {}

for base_name, word_times in alignment_results.items():
    audio_path = None
    for audio_file in os.listdir(AUDIO_FOLDER):
        if os.path.splitext(audio_file)[0] == base_name:
            audio_path = os.path.join(AUDIO_FOLDER, audio_file)
            break
    
    if audio_path and os.path.exists(audio_path):
        output_subdir = os.path.join(SEGMENTS_FOLDER, f"{base_name}_chunks")
        os.makedirs(output_subdir, exist_ok=True)
        
        chunk_info = segment_audio_by_duration(
            audio_path, 
            word_times, 
            output_subdir,
            base_name,
            chunk_duration_sec=10
        )
        
        chunk_segments_metadata[base_name] = chunk_info
        print(f"✓ {base_name}: Created {len(chunk_info)} chunks")

print(f"\n✓ Chunk segmentation complete!")

## Step 10: Create Summary Report and Visualize Results

Generate a report with alignment statistics and create a timeline visualization.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Create summary statistics
print("=" * 60)
print("ALIGNMENT SUMMARY REPORT")
print("=" * 60)

summary_stats = []

for base_name, word_times in alignment_results.items():
    total_duration = word_times[-1]['end'] if word_times else 0
    avg_word_duration = (total_duration / len(word_times)) if word_times else 0
    
    stats = {
        'File': base_name,
        'Total Words': len(word_times),
        'Total Duration (sec)': f"{total_duration:.2f}",
        'Avg Word Duration (ms)': f"{avg_word_duration*1000:.0f}",
        'Words/Minute': f"{(len(word_times) / (total_duration/60)):.0f}" if total_duration > 0 else "N/A"
    }
    summary_stats.append(stats)
    
    print(f"\n{base_name}:")
    for key, value in stats.items():
        if key != 'File':
            print(f"  {key}: {value}")

# Create a DataFrame for better visualization
df_summary = pd.DataFrame(summary_stats)
print("\n" + "=" * 60)
print("SUMMARY TABLE")
print("=" * 60)
print(df_summary.to_string(index=False))

# Visualization: Timeline of words
if alignment_results:
    first_file = list(alignment_results.keys())[0]
    word_times = alignment_results[first_file]
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Plot word durations as bars
    times = [w['start'] for w in word_times]
    durations = [w['duration'] for w in word_times]
    words = [w['word'][:10] for w in word_times]  # Truncate long words
    
    colors = plt.cm.viridis([(d - min(durations)) / (max(durations) - min(durations)) 
                             for d in durations])
    
    bars = ax.bar(range(len(word_times)), durations, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel('Word Index', fontsize=12)
    ax.set_ylabel('Duration (seconds)', fontsize=12)
    ax.set_title(f'Word Duration Timeline - {first_file}', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add colorbar
    sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, 
                               norm=plt.Normalize(vmin=min(durations), vmax=max(durations)))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label('Duration (sec)', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_FOLDER, 'word_duration_timeline.png'), dpi=100, bbox_inches='tight')
    print(f"\n✓ Timeline visualization saved: {os.path.join(OUTPUT_FOLDER, 'word_duration_timeline.png')}")
    plt.show()

# Save detailed report as CSV
report_path = os.path.join(OUTPUT_FOLDER, 'alignment_report.csv')
df_summary.to_csv(report_path, index=False)
print(f"✓ Report saved: {report_path}")

## Step 11: Debug - Check Which Files Failed Alignment

Investigate why only 3 files were processed and identify issues with the other 7 files.

In [ ]:
import os

print("=" * 70)
print("DEBUG: ALIGNMENT FAILURE ANALYSIS")
print("=" * 70)

# List all audio files
print("\n📁 AUDIO FILES FOUND:")
audio_files = {}
for audio_file in os.listdir(AUDIO_FOLDER):
    if audio_file.endswith(('.mp3', '.wav', '.flac')):
        base_name = os.path.splitext(audio_file)[0]
        audio_files[base_name] = audio_file
        print(f"  ✓ {base_name}")

print(f"\nTotal audio files: {len(audio_files)}")

# List all transcription files
print("\n📄 TRANSCRIPTION FILES FOUND:")
transcription_files = {}
for doc_file in os.listdir(TRANSCRIPTION_FOLDER):
    if doc_file.endswith(('.doc', '.docx')):
        base_name = os.path.splitext(doc_file)[0]
        transcription_files[base_name] = doc_file
        print(f"  ✓ {base_name}")

print(f"\nTotal transcription files: {len(transcription_files)}")

# Check for matches
print("\n🔗 FILE MATCHING:")
matched = []
unmatched_audio = []
unmatched_transcription = []

for base_name in audio_files:
    if base_name in transcription_files:
        matched.append(base_name)
        print(f"  ✓ MATCH: {base_name}")
    else:
        unmatched_audio.append(base_name)
        print(f"  ✗ NO TRANSCRIPTION: {base_name}")

for base_name in transcription_files:
    if base_name not in audio_files:
        unmatched_transcription.append(base_name)
        print(f"  ✗ NO AUDIO: {base_name}")

print(f"\n📊 MATCHING SUMMARY:")
print(f"  Matched pairs: {len(matched)}")
print(f"  Unmatched audio files: {len(unmatched_audio)}")
print(f"  Unmatched transcription files: {len(unmatched_transcription)}")

# Check alignment results vs prepared files
print("\n✅ ALIGNMENT RESULTS:")
print(f"  Successfully aligned: {len(alignment_results)}")

successfully_aligned = set(alignment_results.keys())
should_have_aligned = set(matched)

failed_to_align = should_have_aligned - successfully_aligned
print(f"  Failed to align: {len(failed_to_align)}")

if failed_to_align:
    print("\n❌ FILES THAT FAILED ALIGNMENT:")
    for failed_file in sorted(failed_to_align):
        print(f"  - {failed_file}")
        
        # Check if they were prepared in MFA corpus
        corpus_speaker_dir = os.path.join(OUTPUT_FOLDER, "mfa_corpus", failed_file)
        if os.path.exists(corpus_speaker_dir):
            files_in_corpus = os.listdir(corpus_speaker_dir)
            print(f"    Files in corpus: {files_in_corpus}")
            
            # Check transcription content
            txt_files = [f for f in files_in_corpus if f.endswith('.txt')]
            if txt_files:
                txt_path = os.path.join(corpus_speaker_dir, txt_files[0])
                with open(txt_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                print(f"    Transcription preview: {content[:100]}...")
                print(f"    Transcription length: {len(content)} characters")
        else:
            print(f"    ⚠ NOT FOUND in MFA corpus!")

print("\n" + "=" * 70)
print("⚠️  POSSIBLE CAUSES OF ALIGNMENT FAILURES:")
print("=" * 70)
print("""
1. **Transcription quality**: Empty, too short, or too different from audio
2. **Audio quality**: Very poor quality, silent sections, or heavily accented speech
3. **Language model mismatch**: Current model may not fit all speakers/accents
4. **Pronunciation mismatches**: Words not in the phoneme dictionary
5. **Duration mismatch**: Audio and transcription timing severely off

📋 RECOMMENDATION FOR SPANISH MODEL:
Current model: 'spanish_latin_america_mfa'
Alternative options:
- 'spanish_mfa' (Castilian Spanish - more general)
- Create custom dictionary with region-specific terms

Try changing the language model in Step 5 and re-running alignment.
""")


## Step 12: Generate Labels for Audio Chunks

Create text labels/transcriptions for each audio chunk so you have the corresponding text for every audio segment.

In [ ]:
import csv

print("=" * 70)
print("GENERATING LABELS FOR AUDIO CHUNKS")
print("=" * 70)

# Create labels in multiple formats:
# 1. labels.json - Machine readable
# 2. labels.csv - Spreadsheet format
# 3. labels_per_file.txt - Human readable text files

labels_data = []
all_chunks_info = {}

# Function to reconstruct text from word timestamps
def get_text_for_time_range(word_times, start_time, end_time):
    """Extract and concatenate words that fall within a time range."""
    words_in_range = []
    for word_data in word_times:
        # Check if word overlaps with the time range
        if word_data['start'] < end_time and word_data['end'] > start_time:
            words_in_range.append(word_data['word'])
    return " ".join(words_in_range)

print("\nProcessing chunks from chunked segments...\n")

# We need to get the text for each chunk using alignment_results
for base_name, chunk_info_list in chunk_segments_metadata.items():
    
    print(f"Processing {base_name}...")
    
    if base_name not in alignment_results:
        print(f"  ⚠️  No alignment data for {base_name}, skipping labels")
        continue
    
    word_times = alignment_results[base_name]
    file_labels = []
    
    for chunk in chunk_info_list:
        chunk_num = chunk['chunk_num']
        filename = chunk['filename']
        start_time = chunk['start_time']
        end_time = chunk['end_time']
        
        # Get text for this chunk from the alignment data
        chunk_text = get_text_for_time_range(word_times, start_time, end_time)
        
        label_entry = {
            'file_name': base_name,
            'chunk_num': chunk_num,
            'audio_filename': filename,
            'text': chunk_text,
            'start_time': round(start_time, 3),
            'end_time': round(end_time, 3),
            'duration_sec': round(chunk['duration'], 3),
            'word_count': len(chunk_text.split())
        }
        
        labels_data.append(label_entry)
        file_labels.append(label_entry)
    
    all_chunks_info[base_name] = file_labels
    print(f"  ✓ Created {len(file_labels)} labels")

# 1. Save as JSON (machine readable)
print("\n" + "=" * 70)
print("SAVING LABELS IN MULTIPLE FORMATS")
print("=" * 70)

labels_json_path = os.path.join(SEGMENTS_FOLDER, 'labels.json')
with open(labels_json_path, 'w', encoding='utf-8') as f:
    json.dump(labels_data, f, indent=2, ensure_ascii=False)
print(f"\n✓ JSON labels saved: {labels_json_path}")

# 2. Save as CSV (spreadsheet format)
labels_csv_path = os.path.join(SEGMENTS_FOLDER, 'labels.csv')
if labels_data:
    df_labels = pd.DataFrame(labels_data)
    df_labels.to_csv(labels_csv_path, index=False, encoding='utf-8')
    df_labels.to_pickle(labels_csv_path.replace('.csv', '.pkl'))  # Save as pickle for faster loading later
    print(f"✓ CSV labels saved: {labels_csv_path}")

# 3. Save individual text files for each chunk (one label per file)
print("\n📝 Creating individual label files for each audio chunk...")
labels_text_dir = os.path.join(SEGMENTS_FOLDER, 'chunk_labels')
os.makedirs(labels_text_dir, exist_ok=True)

for label_entry in labels_data:
    base_name = label_entry['file_name']
    chunk_num = label_entry['chunk_num']
    audio_filename = label_entry['audio_filename']
    text = label_entry['text']
    
    # Create label filename matching audio filename
    label_filename = audio_filename.replace('.wav', '.txt')
    label_path = os.path.join(labels_text_dir, base_name, label_filename)
    
    # Create subdirectory if needed
    os.makedirs(os.path.dirname(label_path), exist_ok=True)
    
    # Save text label
    with open(label_path, 'w', encoding='utf-8') as f:
        f.write(text)

print(f"✓ Individual label .txt files saved in: {labels_text_dir}")

# 4. Create a human-readable summary
summary_path = os.path.join(SEGMENTS_FOLDER, 'LABELS_SUMMARY.txt')
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write("=" * 80 + "\n")
    f.write("AUDIO CHUNK LABELS SUMMARY\n")
    f.write("=" * 80 + "\n\n")
    
    for base_name, file_labels in all_chunks_info.items():
        f.write(f"\n{'=' * 80}\n")
        f.write(f"FILE: {base_name}\n")
        f.write(f"{'=' * 80}\n\n")
        
        for label in file_labels:
            f.write(f"CHUNK #{label['chunk_num']:03d}\n")
            f.write(f"Audio file: {label['audio_filename']}\n")
            f.write(f"Duration: {label['start_time']:.2f}s - {label['end_time']:.2f}s ({label['duration_sec']:.2f}s)\n")
            f.write(f"Words: {label['word_count']}\n")
            f.write(f"TEXT: {label['text']}\n")
            f.write("-" * 80 + "\n\n")

print(f"✓ Summary saved: {summary_path}")

# Display statistics
print("\n" + "=" * 70)
print("LABELS STATISTICS")
print("=" * 70)
print(f"Total chunks with labels: {len(labels_data)}")
print(f"Files processed: {len(all_chunks_info)}")

if labels_data:
    total_words = sum([l['word_count'] for l in labels_data])
    avg_words_per_chunk = total_words / len(labels_data)
    total_duration = sum([l['duration_sec'] for l in labels_data])
    
    print(f"Total words across all chunks: {total_words}")
    print(f"Average words per chunk: {avg_words_per_chunk:.1f}")
    print(f"Total duration: {total_duration:.1f} seconds")
    
    # Show sample
    print(f"\n📋 SAMPLE CHUNK (first 3 chunks):")
    for i, label in enumerate(labels_data[:3]):
        print(f"\n  Chunk {i+1}: {label['audio_filename']}")
        print(f"  Time: {label['start_time']:.2f}s - {label['end_time']:.2f}s")
        print(f"  Text: \"{label['text'][:80]}...\"" if len(label['text']) > 80 else f"  Text: \"{label['text']}\"")


# Step 13: Generate labels from the original transcription
Before we assigned labels from the processed text (lowercase, no punctuation marks, etc.). We now incorporate the corresponding text from the original transcriptions, preserving all formatting.


In [ ]:
def find_text_in_original(cleaned_text, original_text, tolerance=0.7):
    """
    Find the original text that corresponds to cleaned text.
    Uses fuzzy matching to handle text transformations.
    
    Args:
        cleaned_text: Text from MFA alignment (no punctuation, numbers as words)
        original_text: Original raw text from transcription file
        tolerance: Minimum match ratio (0.0-1.0)
    
    Returns:
        Tuple of (matched_original_text, match_ratio)
    """
    if not cleaned_text.strip():
        return "", 0.0
    
    # Split original text into sentences/phrases for better matching
    original_words = original_text.split()
    cleaned_words = cleaned_text.split()
    
    best_match = ""
    best_ratio = 0.0
    
    # Try to find matching sequences in original text
    for i in range(len(original_words)):
        for j in range(i + 1, min(i + len(cleaned_words) + 5, len(original_words) + 1)):
            candidate = " ".join(original_words[i:j])
            
            # Calculate similarity ratio
            ratio = difflib.SequenceMatcher(
                None, 
                cleaned_text.lower(), 
                candidate.lower()
            ).ratio()
            
            if ratio > best_ratio:
                best_ratio = ratio
                best_match = candidate
    
    # Only return if match quality is good enough
    if best_ratio >= tolerance:
        return best_match, best_ratio
    
    return "", best_ratio

# Reload original (uncleaned) transcriptions for matching
print("=" * 70)
print("ENRICHING LABELS WITH ORIGINAL TRANSCRIPTION TEXT")
print("=" * 70)
print("\nLoading original transcriptions for matching...\n")

original_transcriptions = {}

for doc_file in os.listdir(TRANSCRIPTION_FOLDER):
    if doc_file.endswith(('.doc', '.docx')):
        base_name = os.path.splitext(doc_file)[0]
        doc_path = os.path.join(TRANSCRIPTION_FOLDER, doc_file)
        
        # Load WITHOUT cleaning - keep original punctuation, numbers, formatting
        raw_text = extract_text_from_docx(doc_path)
        original_transcriptions[base_name] = raw_text

print(f"✓ Loaded {len(original_transcriptions)} original transcriptions")

# Enrich labels with original text
print("\nMatching cleaned text to original transcriptions...\n")

enriched_labels = []

for label_entry in labels_data:
    base_name = label_entry['file_name']
    chunk_text_cleaned = label_entry['text']  # From MFA alignment (cleaned)
    
    # Look up original transcription
    if base_name in original_transcriptions:
        original_full_text = original_transcriptions[base_name]
        
        # Find matching text in original
        original_chunk_text, match_ratio = find_text_in_original(
            chunk_text_cleaned, 
            original_full_text,
            tolerance=0.5  # Lower tolerance allows more fuzzy matches
        )
    else:
        original_chunk_text = ""
        match_ratio = 0.0
    
    # Create enriched entry with both texts
    enriched_entry = {
        **label_entry,  # Keep all original fields
        'text_original': original_chunk_text,
        'text_cleaned': chunk_text_cleaned,
        'match_confidence': round(match_ratio, 3)
    }
    
    # Rename 'text' back to clarify it's cleaned
    enriched_labels.append(enriched_entry)

# Update the CSV with original text
labels_csv_path = os.path.join(SEGMENTS_FOLDER, 'labels.csv')
df_labels_enriched = pd.DataFrame(enriched_labels)

# Reorder columns for better readability
column_order = [
    'file_name', 'chunk_num', 'audio_filename',
    'text_cleaned', 'text_original', 'match_confidence',
    'start_time', 'end_time', 'duration_sec', 'word_count'
]
df_labels_enriched = df_labels_enriched[column_order]

# Save enriched version
df_labels_enriched.to_csv(labels_csv_path, index=False, encoding='utf-8')
df_labels_enriched.to_pickle(labels_csv_path.replace('.csv', '.pkl'))

print(f"✓ Enriched labels saved to: {labels_csv_path}")

# Update JSON as well
labels_json_path = os.path.join(SEGMENTS_FOLDER, 'labels.json')
with open(labels_json_path, 'w', encoding='utf-8') as f:
    json.dump(enriched_labels, f, indent=2, ensure_ascii=False)
print(f"✓ Enriched JSON saved to: {labels_json_path}")

# Display statistics
print("\n" + "=" * 70)
print("ENRICHMENT STATISTICS")
print("=" * 70)

match_ratios = [l['match_confidence'] for l in enriched_labels]
print(f"Total chunks: {len(enriched_labels)}")
print(f"Average match confidence: {sum(match_ratios)/len(match_ratios):.3f}")
print(f"Min match confidence: {min(match_ratios):.3f}")
print(f"Max match confidence: {max(match_ratios):.3f}")

# Show samples with both texts
print(f"\n📋 SAMPLE CHUNKS WITH ORIGINAL & CLEANED TEXT:\n")
for i in range(min(3, len(enriched_labels))):
    label = enriched_labels[i]
    print(f"Chunk {i+1}:")
    print(f"  Cleaned (MFA): \"{label['text_cleaned'][:70]}...\"")
    print(f"  Original:     \"{label['text_original'][:70]}...\"")
    print(f"  Match:        {label['match_confidence']:.1%}")
    print()

# Reload as df_x
df_x = df_labels_enriched
print("✓ Updated df_x with enriched labels")
print(f"\nDataframe shape: {df_x.shape}")
print(f"Columns: {list(df_x.columns)}")

# Step 14: Check the final dataset
* "text_cleaned": labels from the processed text.
* "text_original": labels from the original text.

In [ ]:
df_x = pd.read_pickle('/Users/rodrigocarrillo/Downloads/Output/Segmented_audio/labels.pkl')
df_x

# NOTE: The final "labels" dataset with the "text_original" variable will be manually checked for accuracy against the original transcriptions.